## Exploring predictions

In [ ]:
import os
import sys
import gc
import random
import importlib as imp
import numpy as np
import matplotlib.pyplot as plt
# import build_data

import rasterio
from rasterio.windows import Window
from rasterio.transform import Affine

from sklearn.metrics import mean_squared_error, mean_absolute_error

In [ ]:
print(f"python version = {sys.version}")
print(f"numpy version = {np.__version__}")
# print(f"tensorflow version = {tf.__version__}")  

# tf.config.set_visible_devices([], "GPU")  # turn-off tensorflow-metal if it is on
# print(tf.config.list_physical_devices('GPU'))

In [ ]:
SAVE_MODEL_DIRECTORY = "saved_models/"
DATA_DIRECTORY = "data/"
PREDICTIONS_DIRECTORY = "predictions/"
FIGURE_DIRECTORY = "figures/"

In [ ]:
filename = DATA_DIRECTORY + "hfi" + str(2010) + "_merisINT.epsg4326.tif"

with rasterio.open(filename) as orig_tiff:

    lon0, lat0 = orig_tiff.xy(0,0)
    lon1, lat1 = orig_tiff.xy(orig_tiff.shape[0],orig_tiff.shape[1])    
    print(lat0, lat1, lon0, lon1)
    hfi = orig_tiff.read(1)
    meta_data = orig_tiff.meta.copy()

hfi = np.where(hfi!=255, 1., 0.)
plt.imshow(hfi)
plt.colorbar()

In [ ]:
# blend the coastal areas and count those as "non-ocean" areas.

import scipy
hfi_coastal = scipy.ndimage.gaussian_filter(hfi, 5, mode='wrap')
hfi_coastal = np.where(hfi_coastal!=0, 1., 0.)
hfi_coastal = np.asarray(hfi_coastal, dtype="int8")

with rasterio.open(DATA_DIRECTORY + "hfi_coastal_buffer_mask.tif", "w", **meta_data) as dst:
    dst.write(hfi_coastal, 1)


In [ ]:
diff = hfi_coastal - hfi

il, ir = 10000, 10500

plt.figure(figsize=(20,20))
plt.subplot(1,3,1)
plt.imshow(hfi[il:ir, il:ir])

plt.subplot(1,3,2)
plt.imshow(hfi_coastal[il:ir, il:ir])

plt.subplot(1,3,3)
plt.imshow(diff[il:ir, il:ir])

plt.show()